# lltSFT Run Library

Full SFT of SmolLM2-360M with TRL on SmolTalk2 conversations. Two queues on Kaggle's 2x T4, one per GPU.

In [ ]:

import json, os, platform, subprocess, sys, time
T_SESSION = time.time()
print("python", platform.python_version())
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,compute_cap", "--format=csv"], capture_output=True, text=True).stdout)
# Pin the Hugging Face stack only. torch and numpy stay as the image ships them.
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "transformers==5.22.1", "trl==1.17.1", "accelerate==1.15.0",
                "datasets==5.1.0", "tokenizers==0.23.3", "huggingface_hub==2.4.0"], check=False)
print(subprocess.run([sys.executable, "-c", "import torch, transformers, trl, datasets, accelerate; print(torch.__version__, transformers.__version__, trl.__version__, datasets.__version__, accelerate.__version__)"], capture_output=True, text=True).stdout)


In [ ]:
os.makedirs('/tmp/lltsft', exist_ok=True)
COMMON_SRC = '"""Shared code for the lltSFT lab: data selection, the training chat template, the label mask and the evals.\n\nUsed by measure.py (CPU, T1) and worker_sft.py (Kaggle T4, T2), so every run selects the same conversations\nand scores them the same way. The lab page shows the same functions in its build steps.\n\nData: HuggingFaceTB/smoltalk2, config SFT, three no_think splits rebuilt from SmolTalk\'s Smol-Magpie-Ultra,\nSmol-rewrite and Smol-summarize (Apache-2.0 per the SmolTalk card). Streamed in file order, so the selection is\ndeterministic. A conversation is kept when it renders to at most MAX_TOKENS tokens under the SmolLM2 template.\nThe first N_EVAL kept rows of each split are the held-out eval set, the next N_TRAIN the training set.\n"""\nimport hashlib\nimport itertools\nimport json\nimport math\n\nimport torch\n\nBASE_360 = "HuggingFaceTB/SmolLM2-360M"\nBASE_135 = "HuggingFaceTB/SmolLM2-135M"\nINSTRUCT_360 = "HuggingFaceTB/SmolLM2-360M-Instruct"\nINSTRUCT_135 = "HuggingFaceTB/SmolLM2-135M-Instruct"\nDATASET = "HuggingFaceTB/smoltalk2"\nSPLITS = [\n    "smoltalk_smollm3_smol_magpie_ultra_no_think",\n    "smoltalk_smollm3_smol_rewrite_no_think",\n    "smoltalk_smollm3_smol_summarize_no_think",\n]\nMAX_TOKENS = 1024\nN_EVAL = 100\nN_TRAIN = 1200\nN_GEN = 20          # first N_GEN eval rows of each split also give a generation prompt\n\n\ndef to_messages(ex):\n    """SmolTalk2 stores the system prompt in chat_template_kwargs.custom_instructions, not in messages."""\n    msgs = [{"role": m["role"], "content": m["content"]} for m in ex["messages"]]\n    system = (ex.get("chat_template_kwargs") or {}).get("custom_instructions") or ""\n    if system and msgs[0]["role"] != "system":\n        msgs = [{"role": "system", "content": system}] + msgs\n    return msgs\n\n\ndef training_template(instruct_template):\n    """The instruct template with {% generation %} markers around each assistant reply and its <|im_end|>.\n\n    Renders the same text as the instruct template. TRL\'s assistant_only_loss needs the markers.\n    """\n    t = instruct_template.replace(\n        "\' + message[\'content\'] + \'<|im_end|>\' + \'",\n        "\'}}{% if message[\'role\'] == \'assistant\' %}{% generation %}{{ message[\'content\'] + \'<|im_end|>\' }}"\n        "{% endgeneration %}{% else %}{{ message[\'content\'] + \'<|im_end|>\' }}{% endif %}{{\'")\n    assert t != instruct_template, "the instruct template changed; update training_template"\n    return t\n\n\ndef setup_tokenizer(tok, instruct_tok):\n    """Base SmolLM2 ships no chat template: borrow the instruct one, make <|im_end|> the EOS, keep pad apart from EOS."""\n    tok.chat_template = training_template(instruct_tok.chat_template)\n    tok.eos_token = "<|im_end|>"\n    tok.pad_token = "<|endoftext|>"\n    return tok\n\n\ndef header_ids(tok):\n    return tok.convert_tokens_to_ids(["<|im_start|>", "ass", "istant", "Ċ"])\n\n\ndef build_labels(ids, header, im_end):\n    """-100 everywhere except each assistant reply and its <|im_end|>."""\n    labels = [-100] * len(ids)\n    t = 0\n    while t < len(ids):\n        if ids[t:t + len(header)] == header:\n            t += len(header)\n            while t < len(ids):\n                labels[t] = ids[t]\n                t += 1\n                if ids[t - 1] == im_end:\n                    break\n        else:\n            t += 1\n    return labels\n\n\ndef select_conversations(tok, n_eval=N_EVAL, n_train=N_TRAIN, max_tokens=MAX_TOKENS, splits=SPLITS, log=print):\n    """Returns (train, eval, stats). Each item: {"split", "row", "messages", "n_tokens"}."""\n    from datasets import load_dataset\n    train, held, stats = [], [], {}\n    for split in splits:\n        ds = load_dataset(DATASET, "SFT", split=split, streaming=True)\n        kept, seen = [], 0\n        for row, ex in enumerate(ds):\n            seen += 1\n            msgs = to_messages(ex)\n            n = len(tok.apply_chat_template(msgs)["input_ids"])\n            if n <= max_tokens:\n                kept.append({"split": split, "row": row, "messages": msgs, "n_tokens": n})\n                if len(kept) == n_eval + n_train:\n                    break\n        held += kept[:n_eval]\n        train += kept[n_eval:]\n        stats[split] = {"rowsRead": seen, "kept": len(kept), "keptShare": round(len(kept) / seen, 4)}\n        log(split, stats[split])\n    return train, held, stats\n\n\ndef data_hash(items):\n    h = hashlib.sha256()\n    for it in items:\n        h.update(json.dumps(it["messages"], ensure_ascii=False, sort_keys=True).encode("utf-8"))\n    return h.hexdigest()\n\n\n@torch.no_grad()\ndef eval_loss(model, tok, convs, device, batch_size=8, dtype=None):\n    """Token-weighted masked NLL over the assistant tokens of each held-out conversation, one conversation per row."""\n    header, im_end = header_ids(tok), tok.convert_tokens_to_ids("<|im_end|>")\n    was_training = model.training\n    model.eval()\n    nll, count = 0.0, 0\n    for i in range(0, len(convs), batch_size):\n        chunk = convs[i:i + batch_size]\n        ids = [list(tok.apply_chat_template(c["messages"])["input_ids"]) for c in chunk]\n        T = max(len(x) for x in ids)\n        x = torch.full((len(ids), T), tok.pad_token_id, dtype=torch.long)\n        y = torch.full((len(ids), T), -100, dtype=torch.long)\n        am = torch.zeros((len(ids), T), dtype=torch.long)\n        for r, row in enumerate(ids):\n            x[r, :len(row)] = torch.tensor(row)\n            y[r, :len(row)] = torch.tensor(build_labels(row, header, im_end))\n            am[r, :len(row)] = 1\n        x, y, am = x.to(device), y.to(device), am.to(device)\n        with torch.autocast(device.type if hasattr(device, "type") else str(device), dtype=dtype, enabled=dtype is not None):\n            logits = model(input_ids=x, attention_mask=am, use_cache=False).logits\n        tgt = y[:, 1:]\n        m = tgt != -100\n        nll += torch.nn.functional.cross_entropy(logits[:, :-1][m].float(), tgt[m], reduction="sum").item()\n        count += int(m.sum())\n    if was_training:\n        model.train()\n    return {"loss": nll / count, "tokens": count}\n\n\n@torch.no_grad()\ndef eval_generate(model, tok, convs, device, max_new_tokens=512, batch_size=20, dtype=None, template=True):\n    """Greedy answers to the first user turn (with its system prompt). Stop rate = share that end with <|im_end|>."""\n    im_end = tok.convert_tokens_to_ids("<|im_end|>")\n    was_training = model.training\n    model.eval()\n    prompts = []\n    for c in convs:\n        msgs = c["messages"]\n        first = [m for m in msgs[:2] if m["role"] == "system"] + [next(m for m in msgs if m["role"] == "user")]\n        if template:\n            prompts.append(list(tok.apply_chat_template(first, add_generation_prompt=True)["input_ids"]))\n        else:\n            prompts.append(list(tok("\\n\\n".join(m["content"] for m in first) + "\\n\\n")["input_ids"]))\n    out = []\n    for i in range(0, len(prompts), batch_size):\n        chunk = prompts[i:i + batch_size]\n        T = max(len(p) for p in chunk)\n        x = torch.full((len(chunk), T), tok.pad_token_id, dtype=torch.long)\n        am = torch.zeros((len(chunk), T), dtype=torch.long)\n        for r, p in enumerate(chunk):          # left padding for generation\n            x[r, T - len(p):] = torch.tensor(p)\n            am[r, T - len(p):] = 1\n        x, am = x.to(device), am.to(device)\n        with torch.autocast(device.type if hasattr(device, "type") else str(device), dtype=dtype, enabled=dtype is not None):\n            g = model.generate(input_ids=x, attention_mask=am, max_new_tokens=max_new_tokens, do_sample=False,\n                               eos_token_id=im_end, pad_token_id=tok.pad_token_id)\n        for r in range(len(chunk)):\n            new = g[r, T:].tolist()\n            stopped = im_end in new\n            n = new.index(im_end) + 1 if stopped else len([t for t in new if t != tok.pad_token_id])\n            out.append({"stopped": stopped, "newTokens": n, "text": tok.decode(new[:n], skip_special_tokens=False)})\n    if was_training:\n        model.train()\n    stopped = [o for o in out if o["stopped"]]\n    return {\n        "n": len(out), "stopRate": round(len(stopped) / len(out), 4),\n        "meanNewTokens": round(sum(o["newTokens"] for o in out) / len(out), 1),\n        "meanNewTokensStopped": round(sum(o["newTokens"] for o in stopped) / len(stopped), 1) if stopped else None,\n        "samples": out,\n    }\n\n\ndef generation_set(held, n_per_split=N_GEN, splits=SPLITS):\n    return [c for s in splits for c in [h for h in held if h["split"] == s][:n_per_split]]\n'
open('/tmp/lltsft/sft_common.py', 'w').write(COMMON_SRC)
WORKER_SRC = '"""GPU worker for the lltSFT Run Library: full SFT of SmolLM2-360M with TRL on one T4, one run after another.\n\nUsage: python worker_sft.py <queue.json> <data_cache.json> <out_dir>\nCUDA_VISIBLE_DEVICES must name one GPU (SFTTrainer refuses nn.DataParallel). Each run writes <out_dir>/<runId>.json\nas soon as it ends, so a session that dies keeps the finished runs.\n\nRun config keys: runId, label, seed, model, assistantOnly (bool), packing ("bfd" | "naive" | "none"), epochs,\npadIsEos (bool, the broken run), evalEachEpoch (bool), genAtStart (bool), rawPromptGen (bool), lossCheck (bool),\nevalOnly (bool, for the instruct reference).\n"""\nimport gc\nimport json\nimport os\nimport platform\nimport sys\nimport time\nimport traceback\n\nimport torch\n\nHERE = os.path.dirname(os.path.abspath(__file__))\nsys.path.insert(0, HERE)\nimport sft_common as C  # noqa: E402\n\nqueue_path, data_path, out_dir = sys.argv[1:4]\nQUEUE = json.load(open(queue_path))\nDATA = json.load(open(data_path, encoding="utf-8"))\nTRAIN, HELD = DATA["train"], DATA["eval"]\nGEN = C.generation_set(HELD)\nDEV = torch.device("cuda")\nAMP = torch.float16   # T4: capability (7, 5), no bf16 tensor cores\nDRY = os.environ.get("LLT_DRYRUN") == "1"   # CPU smoke test of this script: tiny data, SmolLM2-135M, float32\nif DRY:\n    DEV, AMP = torch.device("cpu"), None\n    TRAIN, HELD = TRAIN[:24], HELD[:6]\n    GEN = HELD[:2]\n    torch.set_num_threads(4)\nGEN_TOKENS = 16 if DRY else 512\n\nimport transformers  # noqa: E402\nimport trl  # noqa: E402\nfrom datasets import Dataset  # noqa: E402\nfrom transformers import AutoModelForCausalLM, AutoTokenizer, TrainerCallback  # noqa: E402\nfrom trl import SFTConfig, SFTTrainer  # noqa: E402\n\nENV = {"python": platform.python_version(), "torch": torch.__version__, "cuda": torch.version.cuda,\n       "transformers": transformers.__version__, "trl": trl.__version__,\n       "datasets": __import__("datasets").__version__, "accelerate": __import__("accelerate").__version__,\n       "gpu": torch.cuda.get_device_name(0) if not DRY else "cpu",\n       "capability": list(torch.cuda.get_device_capability(0)) if not DRY else None}\nprint("env", ENV, flush=True)\n\n\ndef naive_collator(collate):\n    """Same packed rows and labels as TRL\'s collator, but positions never reset, so documents attend to each other."""\n    def wrapped(examples):\n        b = collate(examples)\n        b["position_ids"] = torch.arange(b["input_ids"].shape[1])[None]\n        for k in ("cu_seq_lens_q", "cu_seq_lens_k", "max_length_q", "max_length_k", "seq_idx"):\n            b.pop(k, None)\n        return b\n    return wrapped\n\n\ndef pad_is_eos_collator(collate, pad_id):\n    """The classic bug: pad = EOS, then every label equal to the pad id is set to -100 (DataCollatorForLanguageModeling)."""\n    def wrapped(examples):\n        b = collate(examples)\n        b["labels"][b["labels"] == pad_id] = -100\n        return b\n    return wrapped\n\n\ndef sync():\n    if not DRY:\n        torch.cuda.synchronize()\n\n\nclass Recorder(TrainerCallback):\n    def __init__(self, run, tok):\n        self.run, self.tok, self.evals = run, tok, []\n\n    def on_train_begin(self, args, state, control, model=None, **kw):\n        sync()\n        self.t = time.time()\n\n    def on_epoch_end(self, args, state, control, model=None, **kw):\n        if self.run.get("evalEachEpoch"):\n            sync()\n            t = time.time()\n            e = C.eval_loss(model, self.tok, HELD, DEV, batch_size=1, dtype=AMP)\n            self.evals.append({"epoch": round(state.epoch, 3), "step": state.global_step, "evalLoss": round(e["loss"], 5)})\n            print(self.run["runId"], "epoch eval", self.evals[-1], flush=True)\n            self.t += time.time() - t   # eval time is not training time\n\n    def on_train_end(self, args, state, control, **kw):\n        sync()\n        self.sec = time.time() - self.t\n\n\ndef one(run):\n    t_run = time.time()\n    if not DRY:\n        torch.cuda.reset_peak_memory_stats()\n    rec = {"runId": run["runId"], "module": "lltSFT", "tier": "T2", "label": run["label"],\n           "hardware": {"gpu": ENV["gpu"], "count": 1, "vramGB": None if DRY else round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1)},\n           "env": ENV, "config": run, "seed": run.get("seed")}\n    model_id = run.get("model", C.BASE_360)\n    if DRY:\n        model_id = C.INSTRUCT_135 if run.get("evalOnly") else C.BASE_135\n    if run.get("evalOnly"):\n        tok = AutoTokenizer.from_pretrained(model_id)\n        tok.pad_token = "<|endoftext|>"\n        model = AutoModelForCausalLM.from_pretrained(model_id, dtype=torch.float32).to(DEV)\n        rec["evals"] = {"evalLoss": C.eval_loss(model, tok, HELD, DEV, batch_size=1, dtype=AMP)["loss"]}\n        g = C.eval_generate(model, tok, GEN, DEV, dtype=AMP, max_new_tokens=GEN_TOKENS)\n        rec["evals"]["generation"] = {k: v for k, v in g.items() if k != "samples"}\n        rec["samples"] = g["samples"][:60]\n        rec["wallClockSec"] = round(time.time() - t_run, 1)\n        return rec\n    tok = C.setup_tokenizer(AutoTokenizer.from_pretrained(model_id), AutoTokenizer.from_pretrained(C.INSTRUCT_360))\n    if run.get("padIsEos"):\n        tok.pad_token = tok.eos_token          # the instruct tokenizer ships exactly this: pad = <|im_end|>\n    model = AutoModelForCausalLM.from_pretrained(model_id, dtype=torch.float32).to(DEV)\n    evals = {}\n    if run.get("genAtStart"):\n        evals["base"] = {"evalLoss": C.eval_loss(model, tok, HELD, DEV, batch_size=1, dtype=AMP)["loss"]}\n        g = C.eval_generate(model, tok, GEN, DEV, dtype=AMP, max_new_tokens=GEN_TOKENS)\n        evals["base"]["generation"] = {k: v for k, v in g.items() if k != "samples"}\n        rec["baseSamples"] = g["samples"][:6]\n    packing = run["packing"] != "none"\n    bs = 2 if packing else 4\n    args = SFTConfig(\n        output_dir="/tmp/sft-" + run["runId"], report_to="none", save_strategy="no", eval_strategy="no",\n        fp16=not DRY, bf16=False, use_cpu=DRY, seed=run["seed"], data_seed=run["seed"],\n        learning_rate=5e-5, lr_scheduler_type="cosine", warmup_steps=10, weight_decay=0.0,\n        num_train_epochs=run.get("epochs", 1), per_device_train_batch_size=bs, gradient_accumulation_steps=8,\n        max_length=C.MAX_TOKENS, packing=packing, packing_strategy="bfd",\n        assistant_only_loss=run["assistantOnly"], logging_steps=5, gradient_checkpointing=True,\n        dataloader_num_workers=0,\n    )\n    rcb = Recorder(run, tok)\n    tr = SFTTrainer(model=model, args=args, train_dataset=Dataset.from_list([{"messages": c["messages"]} for c in TRAIN]),\n                    processing_class=tok, callbacks=[rcb])\n    if run["packing"] == "naive":\n        tr.data_collator = naive_collator(tr.data_collator)\n    if run.get("padIsEos"):\n        tr.data_collator = pad_is_eos_collator(tr.data_collator, tok.pad_token_id)\n    rec["trainRows"] = len(tr.train_dataset)\n    if run.get("lossCheck"):\n        # Build it, then prove it: our masked NLL equals TRL\'s compute_loss on the same packed batch.\n        b = next(iter(tr.get_train_dataloader()))\n        b = {k: (v.to(DEV) if torch.is_tensor(v) else v) for k, v in b.items()}\n        model.eval()\n        with torch.no_grad():\n            lib = tr.compute_loss(tr.model, dict(b)).item()\n            logits = model(input_ids=b["input_ids"], position_ids=b["position_ids"], use_cache=False).logits.double()\n            tgt = b["labels"][:, 1:]\n            m = tgt != -100\n            own = torch.nn.functional.cross_entropy(logits[:, :-1][m], tgt[m]).item()\n        model.train()\n        rec["lossCheck"] = {"own": own, "library": lib, "absDiff": abs(own - lib), "dtype": "model float32, own loss in float64",\n                            "batchShape": list(b["input_ids"].shape), "lossTokens": int(m.sum()),\n                            "docsInBatch": int((b["position_ids"] == 0).sum())}\n        print(run["runId"], "lossCheck", rec["lossCheck"], flush=True)\n        del logits\n    if run.get("evalEachEpoch"):\n        rcb.evals.append({"epoch": 0, "step": 0, "evalLoss": round(C.eval_loss(model, tok, HELD, DEV, batch_size=1, dtype=AMP)["loss"], 5)})\n    tr.train()\n    logs = [h for h in tr.state.log_history if "loss" in h]\n    rec["metrics"] = [{"step": h["step"], "trainLoss": h["loss"], "gradNorm": h.get("grad_norm"), "lr": h.get("learning_rate"),\n                       "tokens": h.get("num_tokens"), "tokenAccuracy": h.get("mean_token_accuracy"), "entropy": h.get("entropy")}\n                      for h in logs]\n    tokens = logs[-1].get("num_tokens") if logs else None\n    rec["train"] = {"steps": tr.state.global_step, "trainSec": round(rcb.sec, 1),\n                    "secPerStep": round(rcb.sec / max(1, tr.state.global_step), 3),\n                    "tokensSeen": tokens, "tokensPerSec": round(tokens / rcb.sec, 1) if tokens else None,\n                    "peakMemGB": None if DRY else round(torch.cuda.max_memory_allocated() / 1e9, 2),\n                    "firstLoggedLoss": logs[0]["loss"] if logs else None, "lastLoggedLoss": logs[-1]["loss"] if logs else None}\n    model = tr.model\n    model.eval()\n    model.config.use_cache = True\n    evals["final"] = {"evalLoss": C.eval_loss(model, tok, HELD, DEV, batch_size=1, dtype=AMP)["loss"]}\n    g = C.eval_generate(model, tok, GEN, DEV, dtype=AMP, max_new_tokens=GEN_TOKENS)\n    evals["final"]["generation"] = {k: v for k, v in g.items() if k != "samples"}\n    rec["samples"] = g["samples"]\n    if run.get("rawPromptGen"):\n        g2 = C.eval_generate(model, tok, GEN, DEV, dtype=AMP, template=False, max_new_tokens=GEN_TOKENS)\n        evals["rawPrompt"] = {k: v for k, v in g2.items() if k != "samples"}\n        rec["rawPromptSamples"] = g2["samples"][:6]\n    if rcb.evals:\n        evals["epochs"] = rcb.evals\n    rec["evals"] = evals\n    rec["wallClockSec"] = round(time.time() - t_run, 1)\n    rec["date"] = time.strftime("%Y-%m-%d")\n    return rec\n\n\nfor run in QUEUE:\n    try:\n        rec = one(run)\n        rec["ok"] = True\n    except Exception as e:\n        rec = {"runId": run["runId"], "ok": False, "error": repr(e)[:800], "trace": traceback.format_exc()[-3000:], "config": run}\n        print(rec["trace"], flush=True)\n    with open(os.path.join(out_dir, run["runId"] + ".json"), "w", encoding="utf-8") as f:\n        json.dump(rec, f, indent=1, ensure_ascii=False)\n    print("DONE", run["runId"], rec.get("ok"), rec.get("train"), (rec.get("evals") or {}).get("final", {}).get("evalLoss"), flush=True)\n    gc.collect()\n    if not DRY:\n        torch.cuda.empty_cache()\n'
open('/tmp/lltsft/worker_sft.py', 'w').write(WORKER_SRC)


In [ ]:

t = time.time()
prep = r"""
import json, sys, time
sys.path.insert(0, "/tmp/lltsft")
import sft_common as C
from transformers import AutoTokenizer
tok = C.setup_tokenizer(AutoTokenizer.from_pretrained(C.BASE_360), AutoTokenizer.from_pretrained(C.INSTRUCT_360))
t = time.time()
train, held, stats = C.select_conversations(tok)
json.dump({"train": train, "eval": held, "stats": stats}, open("/tmp/lltsft/data.json", "w"), ensure_ascii=False)
out = {"stats": stats, "trainHash": C.data_hash(train), "evalHash": C.data_hash(held), "nTrain": len(train), "nEval": len(held), "selectSec": round(time.time() - t, 1)}
json.dump(out, open("/kaggle/working/data_selection.json", "w"), indent=1)
print(out)
"""
r = subprocess.run([sys.executable, "-c", prep], capture_output=True, text=True)
print(r.stdout[-3000:]); print(r.stderr[-2000:])
assert r.returncode == 0, "prep failed"
print("prep wall", round(time.time() - t, 1))


In [ ]:
QUEUES = [[{'assistantOnly': True, 'packing': 'bfd', 'epochs': 1, 'seed': 0, 'genAtStart': True, 'rawPromptGen': True, 'lossCheck': True, 'runId': 'lltSFT-T2-assistant-packed-s0', 'label': 'baseline'}, {'assistantOnly': False, 'packing': 'bfd', 'epochs': 1, 'seed': 0, 'runId': 'lltSFT-T2-alltokens-packed-s0', 'label': 'ablation:loss=all-tokens'}, {'assistantOnly': True, 'packing': 'naive', 'epochs': 1, 'seed': 0, 'runId': 'lltSFT-T2-naive-packing-s0', 'label': 'ablation:packing=naive'}, {'assistantOnly': True, 'packing': 'none', 'epochs': 1, 'seed': 0, 'runId': 'lltSFT-T2-padded-s0', 'label': 'ablation:packing=none'}, {'assistantOnly': True, 'packing': 'none', 'epochs': 1, 'seed': 0, 'padIsEos': True, 'runId': 'lltSFT-T2-broken-pad-is-eos-s0', 'label': 'broken:pad-is-eos'}], [{'runId': 'lltSFT-T2-instruct-reference', 'label': 'reference', 'model': 'HuggingFaceTB/SmolLM2-360M-Instruct', 'evalOnly': True}, {'assistantOnly': True, 'packing': 'bfd', 'epochs': 1, 'seed': 1, 'runId': 'lltSFT-T2-assistant-packed-s1', 'label': 'baseline'}, {'assistantOnly': False, 'packing': 'bfd', 'epochs': 1, 'seed': 1, 'runId': 'lltSFT-T2-alltokens-packed-s1', 'label': 'ablation:loss=all-tokens'}, {'assistantOnly': True, 'packing': 'naive', 'epochs': 1, 'seed': 1, 'runId': 'lltSFT-T2-naive-packing-s1', 'label': 'ablation:packing=naive'}, {'assistantOnly': True, 'packing': 'bfd', 'epochs': 3, 'seed': 0, 'evalEachEpoch': True, 'runId': 'lltSFT-T2-epochs3-s0', 'label': 'ablation:epochs=3'}]]

procs = []
for gpu, q in enumerate(QUEUES):
    qp = f"/tmp/lltsft/queue{gpu}.json"; json.dump(q, open(qp, "w"))
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu), PYTHONUNBUFFERED="1", TOKENIZERS_PARALLELISM="false")
    log = open(f"/kaggle/working/worker{gpu}.log", "w")
    procs.append(subprocess.Popen([sys.executable, "/tmp/lltsft/worker_sft.py", qp, "/tmp/lltsft/data.json", "/kaggle/working"],
                                  env=env, stdout=log, stderr=subprocess.STDOUT))
while any(p.poll() is None for p in procs):
    time.sleep(300)
    print(round((time.time() - T_SESSION) / 60, 1), "min", [p.poll() for p in procs], flush=True)
    for gpu in range(len(procs)):
        lines = [l for l in open(f"/kaggle/working/worker{gpu}.log").read().splitlines() if l.startswith(("DONE", "lltSFT", "env"))]
        print(gpu, lines[-1][:300] if lines else "", flush=True)
print("exit codes", [p.returncode for p in procs])
json.dump({"sessionSec": round(time.time() - T_SESSION, 1)}, open("/kaggle/working/session.json", "w"))
print(sorted(os.listdir("/kaggle/working")))
